# 🤟 Türk İşaret Dili (TİD) — Google Colab & Kaggle Model Eğitim Notebook'u

Bu Notebook ile **Google Colab** veya **Kaggle** üzerinde büyük veri setleriyle modelinizi eğitip, çıkan **`model.p`** dosyasını doğrudan bilgisayarınızdaki **`backend/models/model.p`** klasörüne koyarak projede kullanabilirsiniz.

## 1️⃣ Gerekli Kütüphanelerin Kurulumu
Yerel projenizle %100 uyumlu olması için aşağıdaki hücreyi çalıştırın:

In [ ]:
!pip install -q mediapipe opencv-python-headless scikit-learn==1.7.2 numpy

## 2️⃣ Veri Setini Bağlama (Google Drive veya Kaggle)
- **Google Colab kullanıyorsanız:** Veri setinizi Google Drive'a yükleyin veya Kaggle'dan indirin.
- **Kaggle kullanıyorsanız:** Sağ üstten `+ Add Input` diyerek veri setini ekleyin ve `DATA_DIR` yolunu `/kaggle/input/veri-seti-adi` olarak değiştirin.

In [ ]:
import os

# --- SEÇENEK A: Google Drive bağlamak için (Colab) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/data'

# --- SEÇENEK B: Kaggle veya Colab yerel klasörü ---
DATA_DIR = './data'  # Kendi veri seti klasör yolunuzu buraya yazın

print('Veri seti yolu:', os.path.abspath(DATA_DIR))

## 3️⃣ Fotoğraflardan MediaPipe El Koordinatlarını Çıkarma ve `[0, 1]` Normalizasyonu
> **ÖNEMLİ:** Bu hücredeki `normalize_landmarks` fonksiyonu, projenizdeki `backend/app.py` sunucusuyla **birebir aynı matematiksel formülü** kullanır.

In [ ]:
import pickle
import numpy as np
import cv2
import mediapipe as mp

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
FOLDER_NAME_MAP = {
    'C_': 'Ç', 'CH': 'Ç', 'C-': 'Ç',
    'G_': 'Ğ', 'GH': 'Ğ', 'G-': 'Ğ',
    'I_': 'İ', 'II': 'İ', 'I-': 'İ',
    'O_': 'Ö', 'OE': 'Ö', 'O-': 'Ö',
    'S_': 'Ş', 'SH': 'Ş', 'S-': 'Ş',
    'U_': 'Ü', 'UE': 'Ü', 'U-': 'Ü',
}

def normalize_landmarks(x_, y_):
    min_x, max_x = min(x_), max(x_)
    min_y, max_y = min(y_), max(y_)
    width = (max_x - min_x) if (max_x - min_x) > 1e-6 else 1e-6
    height = (max_y - min_y) if (max_y - min_y) > 1e-6 else 1e-6
    data_aux = []
    for x, y in zip(x_, y_):
        data_aux.append((x - min_x) / width)
        data_aux.append((y - min_y) / height)
    return data_aux

def rotate_landmarks(data_aux, angle_deg):
    rad = np.radians(angle_deg)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    rotated = []
    for i in range(0, len(data_aux), 2):
        dx = data_aux[i] - 0.5
        dy = data_aux[i + 1] - 0.5
        rx = dx * cos_a - dy * sin_a + 0.5
        ry = dx * sin_a + dy * cos_a + 0.5
        rotated.append(float(np.clip(rx, 0.0, 1.0)))
        rotated.append(float(np.clip(ry, 0.0, 1.0)))
    return rotated

def augment_landmarks(data_aux):
    augmented = [data_aux.copy()]
    mirrored = []
    for i in range(0, len(data_aux), 2):
        mirrored.append(1.0 - data_aux[i])
        mirrored.append(data_aux[i + 1])
    augmented.append(mirrored)
    for angle in (-10, 10):
        augmented.append(rotate_landmarks(data_aux, angle))
    return augmented

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3, max_num_hands=2)

class_files = {}
for current_root, _, files in os.walk(DATA_DIR):
    imgs = [os.path.join(current_root, f) for f in files if f.lower().endswith(VALID_EXTENSIONS)]
    if not imgs:
        continue
    raw_label = os.path.basename(current_root).strip().upper()
    if raw_label in ('TRAIN', 'TEST', 'VAL', 'VALID', 'VALIDATION', 'DATA', 'DATASET'):
        continue
    label = FOLDER_NAME_MAP.get(raw_label, raw_label)
    class_files.setdefault(label, []).extend(imgs)

data, labels, hand_counts = [], [], []
for label in sorted(class_files.keys()):
    count = 0
    for img_path in class_files[label]:
        stream = np.fromfile(img_path, dtype=np.uint8)
        img = cv2.imdecode(stream, cv2.IMREAD_COLOR)
        if img is None:
            continue
        res = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if not res.multi_hand_landmarks:
            continue
        num_hands = len(res.multi_hand_landmarks)
        x_, y_ = [], []
        for hl in res.multi_hand_landmarks:
            for lm in hl.landmark:
                x_.append(lm.x)
                y_.append(lm.y)
        norm = normalize_landmarks(x_, y_)
        for aug in augment_landmarks(norm):
            data.append(aug)
            labels.append(label)
            hand_counts.append(num_hands)
            count += 1
    print(f'[{label}] -> {count} örnek üretildi')

print('Toplam örnek sayısı:', len(data))

## 4️⃣ Modeli Eğitme ve `model.p` Olarak Kaydetme

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

single_data, single_labels = [], []
two_data, two_labels = [], []

for d, l, h in zip(data, labels, hand_counts):
    arr = np.array(d, dtype=np.float32).flatten()
    if h == 1 and len(arr) == 42:
        single_data.append(arr)
        single_labels.append(l)
    elif h == 2 and len(arr) == 84:
        two_data.append(arr)
        two_labels.append(l)

def create_model():
    return RandomForestClassifier(
        n_estimators=150,
        max_depth=25,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )

models = {}
if len(single_data) > 0:
    X_tr, X_te, y_tr, y_te = train_test_split(
        np.asarray(single_data), np.asarray(single_labels),
        test_size=0.2, stratify=single_labels, random_state=42
    )
    m1 = create_model()
    m1.fit(X_tr, y_tr)
    print(f'Tek El Model Başarısı: {accuracy_score(y_te, m1.predict(X_te))*100:.2f}%')
    models['single_hand'] = m1

if len(two_data) > 0:
    X_tr, X_te, y_tr, y_te = train_test_split(
        np.asarray(two_data), np.asarray(two_labels),
        test_size=0.2, stratify=two_labels, random_state=42
    )
    m2 = create_model()
    m2.fit(X_tr, y_tr)
    print(f'İki El Model Başarısı: {accuracy_score(y_te, m2.predict(X_te))*100:.2f}%')
    models['two_hand'] = m2

with open('model.p', 'wb') as f:
    pickle.dump({'models': models, 'normalized': True}, f)

print('✅ model.p başarıyla oluşturuldu!')

## 5️⃣ Eğitilen `model.p` Dosyasını Bilgisayara İndirme
İnen `model.p` dosyasını bilgisayarınızdaki **`backend/models/model.p`** konumuna yapıştırmanız yeterlidir!

In [ ]:
try:
    from google.colab import files
    files.download('model.p')
except ImportError:
    print('Kaggle kullanıyorsanız sağdaki Output (/kaggle/working/model.p) bölümünden indirebilirsiniz.')